# LayoutVLM: Complete Implementation on Google Colab

**Paper**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193) (CVPR 2025)

**GitHub**: https://github.com/sunfanyunn/LayoutVLM

---

## Overview

This notebook implements the complete LayoutVLM system on Google Colab with full GPU acceleration support. The system generates 3D scene layouts through vision-language model guidance and constraint-based optimization.

### Key Features
- ✅ GPU-accelerated CUDA extensions
- ✅ Blender 4.2.1 LTS integration
- ✅ Automated environment setup
- ✅ Comprehensive error handling

### System Requirements
- **Google Colab**: High RAM runtime recommended
- **GPU**: Tesla T4 or better
- **Storage**: ~5GB for Blender + dependencies + dataset

### Execution Time
- **Setup**: ~15-20 minutes (one-time)
- **Single Scene**: 5-8 minutes (with GPU), 10-15 minutes (CPU fallback)

---

## Getting Started

Execute cells in order from top to bottom. Each step includes clear description and expected output.

**First-time users**: Complete all steps sequentially  
**Returning users**: Can skip Steps 3-7 if already configured

## Step 1: Check GPU Environment

Verify GPU availability and CUDA version.

In [ ]:
!nvidia-smi
!nvcc --version || echo "CUDA Toolkit not installed yet (will install in Step 7)"

## Step 2: Mount Google Drive

Mount Google Drive for persistent storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Install Blender 4.2.1 LTS

Download and install Blender to Google Drive.

**Duration**: ~5 minutes (skipped if already installed)  
**Size**: ~370MB download

In [ ]:
%%bash
BLENDER_DIR="/content/drive/MyDrive/blender-4.2.1-linux-x64"

if [ -d "$BLENDER_DIR" ]; then
    echo "✅ Blender already installed: $BLENDER_DIR"
else
    echo "📦 Downloading Blender 4.2.1 LTS..."
    cd /tmp
    wget -q https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz
    echo "📦 Extracting to Google Drive..."
    tar -xf blender-4.2.1-linux-x64.tar.xz -C /content/drive/MyDrive/
    rm blender-4.2.1-linux-x64.tar.xz
    echo "✅ Blender installed successfully"
fi

# Set environment variable
export BLENDER="/content/drive/MyDrive/blender-4.2.1-linux-x64/blender"
echo "BLENDER=$BLENDER"

## Step 4: Configure GPU Rendering

Verify GPU rendering capabilities in Blender.

In [ ]:
%%bash
BLENDER="/content/drive/MyDrive/blender-4.2.1-linux-x64/blender"

cat > /tmp/check_gpu.py << 'EOF'
import bpy

# Get preferences
prefs = bpy.context.preferences.addons['cycles'].preferences

# Try to enable GPU
prefs.compute_device_type = 'CUDA'
prefs.get_devices()

print("\n=== Available Devices ===")
for device in prefs.devices:
    print(f"  {device.name} ({'GPU' if device.type != 'CPU' else 'CPU'})")

# Enable all CUDA devices
for device in prefs.devices:
    if device.type != 'CPU':
        device.use = True

print("\n✅ GPU rendering configured")
EOF

$BLENDER --background --python /tmp/check_gpu.py

## Step 5: Clone LayoutVLM Repository

Clone the official repository.

In [ ]:
!cd /content && git clone https://github.com/sunfanyunn/LayoutVLM.git
%cd /content/LayoutVLM

## Step 6: Install Python Dependencies

Install required packages in Blender's Python environment.

**Duration**: ~3-5 minutes

In [ ]:
%%bash
PYTHON="/content/drive/MyDrive/blender-4.2.1-linux-x64/4.2/python/bin/python3.11"

echo "📦 Installing Python packages..."
$PYTHON -m pip install --quiet --upgrade pip
$PYTHON -m pip install --quiet torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
$PYTHON -m pip install --quiet openai langchain trimesh shapely scikit-image

echo "✅ All packages installed"

## Step 7: Compile CUDA Extensions (Critical)

Compile GPU-accelerated extensions for optimal performance.

**What this does**:
1. Installs CUDA Toolkit and development headers
2. Fixes Python.h header path
3. Compiles `sort_vertices` CUDA extension
4. Fixes NumPy compatibility

**Duration**: ~8-10 minutes  
**Performance Impact**: 5-10x speedup

In [ ]:
%%bash
set -e

echo "=========================================="
echo "Step 7: Compile CUDA Extensions"
echo "=========================================="
echo ""

cd /content/LayoutVLM/third_party/Rotated_IoU/cuda_op

# Install CUDA Toolkit
echo "📦 Installing CUDA Toolkit..."
apt-get update -qq && apt-get install -y -qq nvidia-cuda-toolkit > /dev/null 2>&1
echo "✅ CUDA Toolkit installed"

# Install Python headers
echo "📦 Installing Python headers..."
apt-get install -y -qq libpython3.11-dev > /dev/null 2>&1
echo "✅ Python headers installed"

# Create symlink
echo "🔗 Creating Python.h symlink..."
BLENDER_PYTHON="/content/drive/MyDrive/blender-4.2.1-linux-x64/4.2/python/include/python3.11"
mkdir -p "$BLENDER_PYTHON"
ln -sf /usr/include/python3.11/* "$BLENDER_PYTHON/"
echo "✅ Symlink created"

# Compile
echo "🔧 Compiling CUDA extension..."
PYTHON="/content/drive/MyDrive/blender-4.2.1-linux-x64/4.2/python/bin/python3.11"
$PYTHON setup.py build_ext --inplace

if [ -f sort_vertices*.so ]; then
    echo "✅ Compilation successful"
    ls -lh sort_vertices*.so
else
    echo "❌ Compilation failed"
    exit 1
fi

# Fix NumPy
echo "🔧 Fixing NumPy compatibility..."
cd ..
sed -i 's/np\.int/int/g' oriented_iou_loss.py
echo "✅ NumPy fixed"

echo ""
echo "=========================================="
echo "✅ CUDA Extension Compilation Complete"
echo "=========================================="

## Step 8: Prepare Dataset

Download scene assets from Objaverse.

**Duration**: ~5-10 minutes  
**Size**: ~2.4GB

In [ ]:
%cd /content/LayoutVLM
!mkdir -p dataset
!gdown --id 1RtixIAhbHGVSs3u2iOZjYuRxQAomXKPp -O dataset/objaverse_holodeck.pkl
!gdown --id 1lbL6bq9hYKwEKOqVVW2QPi5e1FqBzOsE -O dataset/objaverse_holodeck.zip
!cd dataset && unzip -q objaverse_holodeck.zip && rm objaverse_holodeck.zip
!ls -lh dataset/

## Step 9: Configure Qwen API (Alibaba Cloud)

Set up Alibaba Cloud Bailian API credentials.

**Requirements**: 
- Qwen-VL API from Alibaba Cloud Bailian
- Get your API key from: https://bailian.console.aliyun.com/

**Model**: `qwen-vl-max-latest` (OpenAI-compatible format)  
**Performance**: Fast and stable (5-15 seconds per call)

In [ ]:
import os

# Configure Alibaba Cloud Bailian API
os.environ['OPENAI_API_KEY'] = 'sk-your-api-key-here'  # Replace with your API key
os.environ['OPENAI_BASE_URL'] = 'https://dashscope.aliyuncs.com/compatible-mode/v1'

# Model configuration
MODEL_NAME = 'qwen-vl-max-latest'

# Verify configuration
print("✅ Qwen API configured")
print(f"   Base URL: {os.environ['OPENAI_BASE_URL']}")
print(f"   Model: {MODEL_NAME}")
print(f"\n💡 Get your API key from: https://bailian.console.aliyun.com/")

## Step 10: Create Scene Configuration

Define the scene to generate.

**Available scenes**: bedroom, bookstore, buffet_restaurant, children_room, classroom, computer_room, deli, dining_room, florist_shop, game_room, living_room

In [ ]:
scene_type = "living_room"
scene_index = 0

print(f"📝 Scene: {scene_type}_{scene_index}")
print(f"📁 Config: benchmark_tasks/{scene_type}/{scene_type}_{scene_index}.json")

## Step 11: Run LayoutVLM (Main Execution)

Execute the complete pipeline.

**Process**:
1. VLM-guided initialization
2. Constraint-based optimization (GPU-accelerated)
3. 3D rendering with Blender

**Duration**: 5-8 minutes (GPU), 10-15 minutes (CPU fallback)

**Note**: Warnings about `cal_my_giou` are normal (automatic fallback)

In [ ]:
import os
import sys
import subprocess

# Critical Fix 1: matplotlib backend
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']
os.environ['MPLBACKEND'] = 'Agg'

# Critical Fix 2: CUDA extension import path
sys.path.insert(0, "/content/LayoutVLM/third_party/Rotated_IoU")
sys.path.insert(0, "/content/LayoutVLM")

# Configuration
BLENDER = "/content/drive/MyDrive/blender-4.2.1-linux-x64/blender"
task_file = f"benchmark_tasks/{scene_type}/{scene_type}_{scene_index}.json"

# Create execution script
script_content = f'''import sys
sys.path.insert(0, "/content/LayoutVLM/third_party/Rotated_IoU")
sys.path.insert(0, "/content/LayoutVLM")

import os
os.environ['MPLBACKEND'] = 'Agg'

from src.layoutvlm.layoutvlm import run_main

run_main(
    task_file="{task_file}",
    prompt="layoutvlm",
    output_folder="outputs/{scene_type}_{scene_index}",
    data_folder="dataset",
    optimize=True,
    render=True,
    verbose=True
)
'''

with open('/tmp/run_layoutvlm.py', 'w') as f:
    f.write(script_content)

# Critical Fix 3: Manual Xvfb management (preserve GPU access)
print("🚀 Starting LayoutVLM...")
os.environ['DISPLAY'] = ':99'

try:
    # Start Xvfb
    xvfb_process = subprocess.Popen(
        ['Xvfb', ':99', '-screen', '0', '1024x768x24'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    
    # Run Blender (direct execution, no xvfb-run)
    !$BLENDER --background --python /tmp/run_layoutvlm.py
    
finally:
    # Cleanup
    xvfb_process.terminate()
    xvfb_process.wait()

print("✅ Execution complete")

## Step 12: View Results

Display generated scene renders.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
from pathlib import Path

output_dir = f"outputs/{scene_type}_{scene_index}"

# Find all PNG files
image_files = sorted(Path(output_dir).glob("*.png"))

if image_files:
    print(f"📸 Found {len(image_files)} rendered images\n")
    
    # Display images
    for img_path in image_files:
        img = Image.open(img_path)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(img_path.name)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print(f"❌ No images found in {output_dir}")
    print("Check execution output for errors.")

---

## Troubleshooting

### Common Issues

**Issue: "CUDA device not available"**
- Solution: Program uses CPU fallback automatically (slower but functional)
- Check: `Runtime → Change runtime type → GPU`

**Issue: API returns natural language**
- Solution: Automatic retry mechanism handles this (up to 3 attempts)
- Alternative: Use official OpenAI API

**Issue: Runtime disconnected**
- Solution: Keep browser tab active, or use Colab Pro
- Recovery: Re-run from Step 8 (Steps 1-7 persist in Drive)

### Performance

- **Full GPU**: 5-8 minutes per scene
- **Partial GPU**: 8-10 minutes per scene
- **CPU only**: 10-15 minutes per scene

### Getting Help

- GitHub Issues: https://github.com/sunfanyunn/LayoutVLM/issues
- Documentation: See `IMPLEMENTATION_SUMMARY.md`

---

**Paper Citation**:
```
@inproceedings{sun2025layoutvlm,
  title={LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models},
  author={Sun, Fanyue and others},
  booktitle={CVPR},
  year={2025}
}
```